# Q. Bayesian Estimation of a User Ability Parameter from Item Responses

## 1. Probabilistic Model Formulation & Likelihood Function

### A. The Latent Ability and Item Response Model
Let $\theta \in \mathbb{R}$ represent the user's unobserved (latent) ability parameter. Suppose the user responds to a set of $N$ test items. Each item response $X_i$ (for $i = 1, 2, \dots, N$) is modeled as a binary random variable:

$$X_i = \begin{cases} 1, & \text{if the response to item } i \text{ is correct} \\ 0, & \text{if the response to item } i \text{ is incorrect} \end{cases}$$

Conditional on the user's ability $\theta$, the probability of a correct response on item $i$ is defined by an **Item Response Function (IRF)**, denoted $P_i(\theta) = P(X_i = 1 \mid \theta)$.

Under the standard **2-Parameter Logistic (2PL) IRT Model**, this probability is:

$$P_i(\theta) = \sigma\left(a_i(\theta - b_i)\right) = \frac{1}{1 + e^{-a_i(\theta - b_i)}}$$

where:
* $b_i \in \mathbb{R}$ is the **item difficulty parameter** (the level of ability required to have a 50% chance of answering correctly).
* $a_i > 0$ is the **item discrimination parameter** (how steeply the item distinguishes between lower and higher ability users).

Consequently, the probability of an incorrect response is:

$$1 - P_i(\theta) = P(X_i = 0 \mid \theta) = \frac{e^{-a_i(\theta - b_i)}}{1 + e^{-a_i(\theta - b_i)}}$$


### B. Derivation of the Joint Likelihood Function
Under the fundamental assumption of **local independence**, the user's responses to different items are conditionally independent given their ability parameter $\theta$.

Therefore, given an observed response pattern vector $\mathbf{x} = (x_1, x_2, \dots, x_N)^T \in \{0, 1\}^N$, the joint likelihood function $L(\mathbf{x} \mid \theta) = P(\mathbf{X} = \mathbf{x} \mid \theta)$ is the product of the individual item probabilities:

$$L(\mathbf{x} \mid \theta) = \prod_{i=1}^N P_i(\theta)^{x_i} \left(1 - P_i(\theta)\right)^{1 - x_i}$$

Taking the natural logarithm yields the **log-likelihood function**:

$$\ell(\theta) = \log L(\mathbf{x} \mid \theta) = \sum_{i=1}^N \left[ x_i \log P_i(\theta) + (1 - x_i) \log \left(1 - P_i(\theta)\right) \right]$$


### C. Prior Distribution and Unnormalized Posterior
To frame the estimation within a Bayesian framework, we assign a prior distribution $p(\theta)$ to the user ability. The standard choice is a normal prior reflecting the population distribution:

$$\theta \sim \mathcal{N}(\mu_0, \sigma_0^2) \implies p(\theta) = \frac{1}{\sqrt{2\pi\sigma_0^2}} \exp\left( -\frac{(\theta - \mu_0)^2}{2\sigma_0^2} \right)$$

*(Typically $\mu_0 = 0$ and $\sigma_0^2 = 1$ for a standardized ability scale).*

By **Bayes' Theorem**, the posterior distribution of the user ability $\theta$ given the observed response vector $\mathbf{x}$ is:

$$p(\theta \mid \mathbf{x}) = \frac{P(\mathbf{x} \mid \theta) \, p(\theta)}{P(\mathbf{x})} = \frac{\prod_{i=1}^N P_i(\theta)^{x_i} \left(1 - P_i(\theta)\right)^{1 - x_i} \cdot p(\theta)}{\int_{-\infty}^{\infty} P(\mathbf{x} \mid \theta') \, p(\theta') \, d\theta'}$$

In unnormalized form, the posterior density satisfies:

$$p(\theta \mid \mathbf{x}) \propto \left( \prod_{i=1}^N P_i(\theta)^{x_i} \left(1 - P_i(\theta)\right)^{1 - x_i} \right) \cdot \exp\left( -\frac{(\theta - \mu_0)^2}{2\sigma_0^2} \right)$$

In [ ]:
import numpy as np
import plotly.graph_objects as go
from scipy.stats import norm

def run_task_1_bayesian_ability_plot():
    """
    Task 1: Generate an interactive visualization of the Prior,
    Normalized Likelihood, and Posterior distributions for
    estimating a user's latent ability (theta) from binary item responses.
    """
    # 1. Parameter Initialization
    # Response vector x: 1 = Correct, 0 = Incorrect
    responses = np.array([1, 1, 0, 1, 0])

    # Item parameters for 2PL IRT Model
    difficulties = np.array([-1.5, -0.5, 0.0, 0.8, 1.5])  # Item difficulty (b_i)
    discriminations = np.array([1.2, 0.8, 1.5, 1.0, 1.1]) # Item discrimination (a_i)

    # Prior hyperparameters (Standard Normal Prior ~ N(0, 1))
    prior_mean = 0.0
    prior_std = 1.0

    # Continuous domain grid for Latent Ability Parameter (theta)
    theta = np.linspace(-4, 4, 1000)

    # 2. Prior Distribution p(theta)
    prior = norm.pdf(theta, loc=prior_mean, scale=prior_std)

    # 3. Compute Joint Log-Likelihood and Likelihood L(x | theta)
    log_likelihood = np.zeros_like(theta)

    for x_i, b_i, a_i in zip(responses, difficulties, discriminations):
        # 2-Parameter Logistic (2PL) Item Response Function
        p_i = 1 / (1 + np.exp(-a_i * (theta - b_i)))

        # Clip probabilities to prevent log(0) numerical instability
        p_i = np.clip(p_i, 1e-12, 1.0 - 1e-12)

        # Accumulate log-likelihood
        log_likelihood += x_i * np.log(p_i) + (1 - x_i) * np.log(1 - p_i)

    likelihood = np.exp(log_likelihood)

    # Normalize likelihood curve so it shares the same area scale (PDF format) as Prior/Posterior
    likelihood_normalized = likelihood / np.trapz(likelihood, theta)

    # 4. Compute Posterior Distribution p(theta | x)
    unnormalized_posterior = likelihood * prior
    posterior = unnormalized_posterior / np.trapz(unnormalized_posterior, theta)

    # 5. Extract Point Estimates
    # Maximum A Posteriori (MAP) - Posterior Mode
    map_index = np.argmax(posterior)
    theta_map = theta[map_index]

    # Expected A Posteriori (EAP) - Posterior Mean
    theta_eap = np.trapz(theta * posterior, theta)

    # 6. Construct Interactive Plot
    fig = go.Figure()

    # Prior curve
    fig.add_trace(go.Scatter(
        x=theta, y=prior,
        mode='lines',
        name='Prior p(θ)',
        line=dict(color='gray', dash='dash', width=2)
    ))

    # Likelihood curve
    fig.add_trace(go.Scatter(
        x=theta, y=likelihood_normalized,
        mode='lines',
        name='Normalized Likelihood L(x|θ)',
        line=dict(color='#1f77b4', width=2)
    ))

    # Posterior curve
    fig.add_trace(go.Scatter(
        x=theta, y=posterior,
        mode='lines',
        name='Posterior p(θ|x)',
        line=dict(color='#d62728', width=3)
    ))

    # Vertical marker for MAP
    fig.add_vline(
        x=theta_map, line_width=1.5, line_dash="dot", line_color="#d62728",
        annotation_text=f"MAP = {theta_map:.2f}", annotation_position="top left"
    )

    # Vertical marker for EAP
    fig.add_vline(
        x=theta_eap, line_width=1.5, line_dash="dashdot", line_color="green",
        annotation_text=f"EAP = {theta_eap:.2f}", annotation_position="top right"
    )

    # Layout styling
    fig.update_layout(
        title="<b>Task 1: Bayesian Estimation of User Ability Parameter (θ)</b>",
        xaxis_title="<b>User Ability Parameter (θ)</b>",
        yaxis_title="<b>Probability Density</b>",
        template="plotly_white",
        legend=dict(x=0.02, y=0.98),
        hovermode="x unified"
    )

    fig.show()

# Run the task
run_task_1_bayesian_ability_plot()

/tmp/ipykernel_770/3232392687.py:45: DeprecationWarning:

`trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.

/tmp/ipykernel_770/3232392687.py:49: DeprecationWarning:

`trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.

/tmp/ipykernel_770/3232392687.py:57: DeprecationWarning:

`trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.



---
## 2: Sequential Likelihood Contribution & Joint History Likelihood

### 1. Likelihood Contribution of a Single Response at Step $k$
Let $y_k \in \{0, 1\}$ represent the user's response to item $k$ at step $k$, where $y_k = 1$ denotes a correct response and $y_k = 0$ denotes an incorrect response[cite: 1]. Under the 2-Parameter Logistic (2PL) Item Response Theory model, the probability of a correct response given latent ability $\Theta = \theta$ is[cite: 1]:

$$p_k(\theta) = P(Y_k = 1 \mid \Theta = \theta) = \frac{1}{1 + e^{-a_k(\theta - b_k)}}$$

where $a_k > 0$ is the known item discrimination parameter and $b_k \in \mathbb{R}$ is the known item difficulty parameter[cite: 1].

The likelihood contribution $L(y_k \mid \theta)$ of the single isolated observation $y_k$ at step $k$ is expressed as a Bernoulli probability mass function:

$$L(y_k \mid \theta) = P(Y_k = y_k \mid \Theta = \theta) = \left[ p_k(\theta) \right]^{y_k} \left[ 1 - p_k(\theta) \right]^{1 - y_k}$$

Substituting $p_k(\theta)$ explicitly into the Bernoulli likelihood gives:

$$L(y_k \mid \theta) = \left( \frac{1}{1 + e^{-a_k(\theta - b_k)}} \right)^{y_k} \left( \frac{e^{-a_k(\theta - b_k)}}{1 + e^{-a_k(\theta - b_k)}} \right)^{1 - y_k}$$


### 2. Joint Likelihood Function for the Running History Vector $y^{(k)}$
Let $y^{(k)} = (y_1, y_2, \dots, y_k)$ represent the running vector of observed responses accumulated up to step $k$ (where $1 \le k \le n$)[cite: 1].

Assuming **local independence** (conditional on latent ability $\theta$, responses to different items are conditionally independent), the joint likelihood function $L\left(y^{(k)} \mid \theta\right)$ is the product of individual item likelihood contributions from step $1$ to step $k$:

$$L\left(y^{(k)} \mid \theta\right) = P\left(Y^{(k)} = y^{(k)} \mid \Theta = \theta\right) = \prod_{i=1}^k L(y_i \mid \theta)$$

$$\implies L\left(y^{(k)} \mid \theta\right) = \prod_{i=1}^k \left[ p_i(\theta) \right]^{y_i} \left[ 1 - p_i(\theta) \right]^{1 - y_i}$$

Substituting the 2PL Item Response Function into the product yields:

$$L\left(y^{(k)} \mid \theta\right) = \prod_{i=1}^k \left( \frac{1}{1 + e^{-a_i(\theta - b_i)}} \right)^{y_i} \left( \frac{e^{-a_i(\theta - b_i)}}{1 + e^{-a_i(\theta - b_i)}} \right)^{1 - y_i}$$

---
## 3: Mathematical Formulation of the Running Update

By Bayes' Theorem, the posterior probability density function of the user's latent ability $\Theta = \theta$ at step $k$, given the running history vector of responses $y^{(k)} = (y_1, y_2, \dots, y_k)$, is proportional to the product of the joint likelihood of $y^{(k)}$ and the initial prior density $f_{\Theta}^{(0)}(\theta)$[cite: 1]:

$$f_{\Theta \mid Y^{(k)}}\left(\theta \mid y^{(k)}\right) \propto L\left(y^{(k)} \mid \theta\right) \cdot f_{\Theta}^{(0)}(\theta)$$

Under the assumption of local independence, the joint likelihood factors as[cite: 1]:

$$L\left(y^{(k)} \mid \theta\right) = \left( \prod_{i=1}^{k-1} L(y_i \mid \theta) \right) \cdot L(y_k \mid \theta) = L\left(y^{(k-1)} \mid \theta\right) \cdot L(y_k \mid \theta)$$

Substituting this product into the Bayes' formula for step $k$:

$$f_{\Theta \mid Y^{(k)}}\left(\theta \mid y^{(k)}\right) \propto L(y_k \mid \theta) \cdot \underbrace{\left[ L\left(y^{(k-1)} \mid \theta\right) \cdot f_{\Theta}^{(0)}(\theta) \right]}_{\propto f_{\Theta \mid Y^{(k-1)}}\left(\theta \mid y^{(k-1)}\right)}$$

Notice that the term in brackets is proportional to the posterior density from the previous step, $f_{\Theta \mid Y^{(k-1)}}\left(\theta \mid y^{(k-1)}\right)$[cite: 1].

This yields the **recursive Bayesian update relationship** (up to a normalizing constant):

$$f_{\Theta \mid Y^{(k)}}\left(\theta \mid y^{(k)}\right) \propto f_{\Theta \mid Y^{(k-1)}}\left(\theta \mid y^{(k-1)}\right) \cdot L(y_k \mid \theta)$$

Substituting the explicit 2PL Bernoulli likelihood contribution $L(y_k \mid \theta)$ derived in Task 2[cite: 1]:

$$f_{\Theta \mid Y^{(k)}}\left(\theta \mid y^{(k)}\right) \propto f_{\Theta \mid Y^{(k-1)}}\left(\theta \mid y^{(k-1)}\right) \cdot \left( \frac{1}{1 + e^{-a_k(\theta - b_k)}} \right)^{y_k} \left( \frac{e^{-a_k(\theta - b_k)}}{1 + e^{-a_k(\theta - b_k)}} \right)^{1 - y_k}$$

### Normalized Recursive Expression
To express this as an exact probability density function, we normalize by integrating over the entire domain of $\theta$:

$$f_{\Theta \mid Y^{(k)}}\left(\theta \mid y^{(k)}\right) = \frac{f_{\Theta \mid Y^{(k-1)}}\left(\theta \mid y^{(k-1)}\right) \cdot L(y_k \mid \theta)}{\int_{-\infty}^{\infty} f_{\Theta \mid Y^{(k-1)}}\left(\theta' \mid y^{(k-1)}\right) \cdot L(y_k \mid \theta') \, d\theta'}$$

This formulation demonstrates that the posterior distribution at step $k-1$ acts directly as the prior distribution for step $k$[cite: 1].

---
## 4: Dynamic Shifting Mechanics of a Correct Response to a High-Difficulty Item

When a user correctly answers ($y_k = 1$) an item with a high difficulty parameter (large $b_k$), the running posterior probability density $f_{\Theta \mid Y^{(k)}}\left(\theta \mid y^{(k)}\right)$ shifts significantly toward higher ability values[cite: 1]. Mathematically, this behavior can be understood through three key components:

### 1. Likelihood Weighting Function
For a correct response ($y_k = 1$), the single-step likelihood contribution $L(y_k = 1 \mid \theta)$ simplifies to the 2PL Item Response Function[cite: 1]:

$$L(y_k = 1 \mid \theta) = p_k(\theta) = \frac{1}{1 + e^{-a_k(\theta - b_k)}}$$

Because $b_k$ is large, $p_k(\theta)$ remains near zero for lower values of $\theta$ and rapidly increases toward $1.0$ as $\theta$ crosses above $b_k$.

### 2. Multiplicative Shift on the Prior State
In the recursive update, the new posterior density is formed by point-wise multiplying the prior state $f_{\Theta \mid Y^{(k-1)}}\left(\theta \mid y^{(k-1)}\right)$ by this likelihood curve[cite: 1]:

$$f_{\Theta \mid Y^{(k)}}\left(\theta \mid y^{(k)}\right) \propto f_{\Theta \mid Y^{(k-1)}}\left(\theta \mid y^{(k-1)}\right) \cdot \frac{1}{1 + e^{-a_k(\theta - b_k)}}$$

Because the likelihood curve $p_k(\theta)$ acts as a monotonic weighting function:
* **Suppression of Lower Ability Domain ($\theta < b_k$):** For ability levels below the item difficulty, $p_k(\theta)$ is very small, severely dampening the left tail of the prior density.
* **Amplification of Higher Ability Domain ($\theta > b_k$):** For ability levels exceeding the item difficulty, $p_k(\theta)$ approaches $1.0$, preserving or accentuating the right tail of the prior distribution.

### 3. Posterior Peak Shift (MAP and EAP)
The point-wise multiplication penalizes the lower region of the prior density far more heavily than the upper region. When the resulting product curve is renormalized, the mode (peak) of the distribution, $\hat{\theta}_{\text{MAP}}^{(k)}$, as well as the expected mean, $\hat{\theta}_{\text{Bayes}}^{(k)}$, are forced to move rightward along the $\theta$-axis relative to the step $k-1$ peak.

In intuitive psychometric terms, correctly solving a hard question provides strong empirical evidence that the user's ability is higher than previously assumed, triggering a bold positive update in the platform's belief.

---
## 5: Tracking Certainty and Sharpness via the Discrimination Parameter $a_k$

The discrimination parameter $a_k > 0$ governs the slope of the 2PL Item Response Function $p_k(\theta) = \frac{1}{1 + e^{-a_k(\theta - b_k)}}$ at its inflection point $\theta = b_k$[cite: 1]. During a running Bayesian update, $a_k$ directly controls the magnitude of information extracted from a response, thereby dictating the variance (or "sharpness") of the resulting posterior distribution $f_{\Theta \mid Y^{(k)}}\left(\theta \mid y^{(k)}\right)$[cite: 1].


### 1. Mathematical Mechanism

In the recursive update, the log-posterior density is given by[cite: 1]:

$$\log f_{\Theta \mid Y^{(k)}}\left(\theta \mid y^{(k)}\right) = \log f_{\Theta \mid Y^{(k-1)}}\left(\theta \mid y^{(k-1)}\right) + \log L(y_k \mid \theta) + C$$

The curvature (second derivative with respect to $\theta$) of the log-posterior determines the variance/uncertainty of the distribution around its peak (Fisher Information):

$$I_k(\theta) = -\frac{\partial^2}{\partial \theta^2} \log f_{\Theta \mid Y^{(k)}}\left(\theta \mid y^{(k)}\right) = I_{k-1}(\theta) + a_k^2 \cdot p_k(\theta)\left(1 - p_k(\theta)\right)$$

Because the information added at step $k$ scales with $a_k^2$, higher discrimination values inject substantially more curvature into the log-posterior, forcing the variance to decrease rapidly.


### 2. High Discrimination ($a_k \gg 0$, Very Large)

* **Behavior of Likelihood:** As $a_k \to \infty$, the likelihood $p_k(\theta)$ approaches a sharp, step-like threshold function jumping from $0$ to $1$ at $\theta = b_k$.
* **Impact on Posterior:** Point-wise multiplying by this steep curve drastically truncates one side of the prior density while preserving the other.
* **Sharpness & Certainty:** The posterior variance drops significantly, resulting in a **narrower, much sharper peak**. The system gains a high boost in measurement confidence from a single observation because the item cleanly distinguishes whether $\theta > b_k$ or $\theta < b_k$.


### 3. Low Discrimination ($a_k \approx 0$, Very Small)

* **Behavior of Likelihood:** As $a_k \to 0$, $p_k(\theta) \approx 0.5$ across nearly the entire domain of $\theta$, representing an uninformative item (guessing/flat response).
* **Impact on Posterior:** Multiplying the prior $f_{\Theta \mid Y^{(k-1)}}\left(\theta \mid y^{(k-1)}\right)$ by an almost constant likelihood value ($0.5$) leaves the shape of the distribution virtually unchanged after normalization.
* **Sharpness & Certainty:** The posterior distribution retains its existing variance and broad shape. Almost **no new certainty or sharpness is gained**, indicating that the response provided negligible empirical evidence about the user's true ability.

---
## 6: Numerical Implementation of a Running Grid & Sequential Normalization

Because the 2PL Item Response Theory likelihood function $L(y_k \mid \theta)$ is non-linear and non-conjugate with a Normal prior, the posterior density $f_{\Theta \mid Y^{(k)}}\left(\theta \mid y^{(k)}\right)$ cannot be solved analytically in closed form[cite: 1]. We maintain and update the running posterior density numerically over a discrete grid[cite: 1].


### Algorithmic Step-by-Step Procedure

#### Step 1: Discretize the Latent Ability Domain ($\theta$)
* Define a bounded, fine-grained grid of $M$ equally spaced points across the plausible domain of ability (e.g., $M = 1000$ points from $\theta_{\min} = -4.0$ to $\theta_{\max} = 4.0$).
#### Step 1: Discretize the Latent Ability Domain ($\theta$)
* Define a bounded, fine-grained grid of $M$ equally spaced points across the plausible domain of ability (e.g., $M = 1000$ points from $\theta_{\min} = -4.0$ to $\theta_{\max} = 4.0$).
* Store this coordinate array as $\boldsymbol{\theta} = [\theta_1, \theta_2, \dots, \theta_M]$.
* Calculate the constant grid step size: $\Delta \theta = \frac{\theta_{\max} - \theta_{\min}}{M - 1}$.
* Calculate the constant grid step size: $\Delta \theta = \frac{\theta_{\max} - \theta_{\min}}{M - 1}$.

#### Step 2: Initialize the Base Prior Density State ($k = 0$)
* Evaluate the Standard Normal prior density $f_{\Theta}^{(0)}(\theta) \sim \mathcal{N}(0, 1)$ across all grid points[cite: 1]:

$$f^{(0)}_m = \frac{1}{\sqrt{2\pi}} \exp\left(-\frac{\theta_m^2}{2}\right), \quad \text{for } m = 1, 2, \dots, M$$

* Normalize this initial vector using numerical integration (e.g., Trapezoidal rule) so that the total area sums to $1.0$:

$$\text{Area}^{(0)} = \sum_{m=1}^{M-1} \left( \frac{f^{(0)}_m + f^{(0)}_{m+1}}{2} \right) \Delta \theta, \quad \text{Set } \mathbf{f}^{(0)} = \frac{\mathbf{f}^{(0)}}{\text{Area}^{(0)}}$$

#### Step 3: Sequential Point-Wise Update ($k = 1, 2, \dots, n$)
Upon observing a new response $y_k \in \{0, 1\}$ for an item with parameters $a_k$ and $b_k$[cite: 1]:

1. **Evaluate the Item Likelihood Vector:** Compute the single-step 2PL likelihood contribution across every grid point $\theta_m$[cite: 1]:

$$p_k(\theta_m) = \frac{1}{1 + e^{-a_k(\theta_m - b_k)}}$$

$$L(y_k \mid \theta_m) = [p_k(\theta_m)]^{y_k} [1 - p_k(\theta_m)]^{1 - y_k}, \quad \text{for } m = 1, 2, \dots, M$$

2. **Compute Unnormalized Posterior Vector:** Perform point-wise multiplication between the likelihood vector and the previous step's normalized posterior vector $\mathbf{f}^{(k-1)}$[cite: 1]:

$$\tilde{f}^{(k)}_m = f^{(k-1)}_m \cdot L(y_k \mid \theta_m), \quad \text{for } m = 1, 2, \dots, M$$

3. **Sequential Computational Normalization:** Perform numerical integration over the unnormalized grid $\tilde{\mathbf{f}}^{(k)}$ using the Composite Trapezoidal Rule (`np.trapezoid` or `scipy.integrate.trapezoid`) to evaluate the marginal likelihood normalizing constant $Z_k$:

$$Z_k = \int_{\theta_{\min}}^{\theta_{\max}} \tilde{f}^{(k)}(\theta) \, d\theta \approx \frac{\Delta \theta}{2} \left[ \tilde{f}^{(k)}_1 + 2 \sum_{m=2}^{M-1} \tilde{f}^{(k)}_m + \tilde{f}^{(k)}_M \right]$$

4. **Update the State Vector:** Divide every grid element by $Z_k$ to produce the fully normalized posterior vector for step $k$:

$$f^{(k)}_m = \frac{\tilde{f}^{(k)}_m}{Z_k}, \quad \text{for } m = 1, 2, \dots, M$$

This normalized vector $\mathbf{f}^{(k)}$ becomes the prior distribution state for step $k + 1$[cite: 1].

In [ ]:
import numpy as np
import plotly.graph_objects as go
from scipy.stats import norm

def simulate_bayesian_ability_tracking(
    theta_true=0.75,
    n_items=20,
    grid_points=1000,
    seed=42
):
    """
    Task 7: Simulates user responses to n_items, updates the posterior density
    on a grid, tracks posterior mean (EAP) and MAP estimates over time,
    and visualizes estimator convergence relative to theta_true.
    """
    np.random.seed(seed)

    # 1. Define Fixed Ability Grid & Initialization
    theta_grid = np.linspace(-4.0, 4.0, grid_points)
    delta_theta = theta_grid[1] - theta_grid[0]

    # Initial Standard Normal Prior ~ N(0, 1)
    prior = norm.pdf(theta_grid, loc=0.0, scale=1.0)
    running_posterior = prior / np.trapz(prior, theta_grid)

    # Store initial step 0 estimates
    eap_history = [np.trapz(theta_grid * running_posterior, theta_grid)]
    map_history = [theta_grid[np.argmax(running_posterior)]]

    # 2. Sequential Simulation over n_items
    for k in range(1, n_items + 1):
        # Draw random item parameters
        b_k = np.random.normal(0.0, 1.0)          # Difficulty b_k ~ N(0, 1)
        a_k = np.random.uniform(0.5, 2.0)         # Discrimination a_k ~ Uniform(0.5, 2.0)

        # Calculate true response probability p_k(theta_true)
        p_true = 1.0 / (1.0 + np.exp(-a_k * (theta_true - b_k)))

        # Simulate response y_k in {0, 1}
        u = np.random.uniform(0.0, 1.0)
        y_k = 1 if u < p_true else 0

        # Evaluate likelihood L(y_k | theta) over the entire grid
        p_grid = 1.0 / (1.0 + np.exp(-a_k * (theta_grid - b_k)))
        p_grid = np.clip(p_grid, 1e-12, 1.0 - 1e-12)
        likelihood_k = (p_grid**y_k) * ((1.0 - p_grid)**(1.0 - y_k))

        # Unnormalized recursive update: Prior_k * Likelihood_k
        unnormalized_posterior = running_posterior * likelihood_k

        # Sequential Normalization using Trapezoidal Rule
        normalizing_constant = np.trapz(unnormalized_posterior, theta_grid)
        running_posterior = unnormalized_posterior / normalizing_constant

        # Compute & store running point estimators
        # MAP (Posterior Mode)
        map_est = theta_grid[np.argmax(running_posterior)]
        # EAP (Posterior Mean)
        eap_est = np.trapz(theta_grid * running_posterior, theta_grid)

        map_history.append(map_est)
        eap_history.append(eap_est)

    # 3. Interactive Visualization using Plotly
    steps = list(range(n_items + 1))
    fig = go.Figure()

    # Posterior Mean (EAP) curve
    fig.add_trace(go.Scatter(
        x=steps, y=eap_history,
        mode='lines+markers',
        name='Posterior Mean (θ_EAP)',
        line=dict(color='#2CA02C', width=2.5),
        marker=dict(size=6)
    ))

    # MAP estimate curve
    fig.add_trace(go.Scatter(
        x=steps, y=map_history,
        mode='lines+markers',
        name='MAP Estimate (θ_MAP)',
        line=dict(color='#D62728', width=2, dash='dot'),
        marker=dict(size=6)
    ))

    # Static Reference Line at theta_true
    fig.add_hline(
        y=theta_true, line_width=2, line_dash="dash", line_color="black",
        annotation_text=f"True Ability (θ_true = {theta_true})",
        annotation_position="bottom right"
    )

    fig.update_layout(
        title="<b>Task 7: Sequential Convergence of User Ability Estimators (n = 20)</b>",
        xaxis_title="<b>Item Sequence Step (k)</b>",
        yaxis_title="<b>Estimated Ability Parameter (θ)</b>",
        template="plotly_white",
        xaxis=dict(dtick=2),
        hovermode="x unified"
    )

    fig.show()

# Execute the simulation
simulate_bayesian_ability_tracking()

/tmp/ipykernel_770/596005582.py:24: DeprecationWarning:

`trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.

/tmp/ipykernel_770/596005582.py:27: DeprecationWarning:

`trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.

/tmp/ipykernel_770/596005582.py:52: DeprecationWarning:

`trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.

/tmp/ipykernel_770/596005582.py:59: DeprecationWarning:

`trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.



---
# Q. Bayesian Tracking of Click-Through Rates (CTR) via Conjugate Beta-Binomial Updates

In [ ]:
import numpy as np
import plotly.graph_objects as go
from scipy.stats import beta

def plot_beta_ctr_priors():
    """
    Task 1: Plot the PDF of a Beta(alpha, beta) distribution for three distinct parameter pairs:
    1. Uninformative state: (alpha=1, beta=1)
    2. Right-skewed state: (alpha=2, beta=8)
    3. Left-skewed state: (alpha=8, beta=2)
    """
    # 1. Define continuous domain theta in [0, 1]
    theta = np.linspace(0.001, 0.999, 1000)

    # 2. Define parameter pairs
    shapes = [
        (1, 1, 'Uninformative state: (α=1, β=1)', 'gray', 'dash'),
        (2, 8, 'Right-skewed state: (α=2, β=8)', '#1f77b4', 'solid'),
        (8, 2, 'Left-skewed state: (α=8, β=2)', '#d62728', 'solid')
    ]

    fig = go.Figure()

    # 3. Compute and plot PDFs
    for a, b, label, color, dash in shapes:
        pdf_vals = beta.pdf(theta, a, b)
        fig.add_trace(go.Scatter(
            x=theta,
            y=pdf_vals,
            mode='lines',
            name=label,
            line=dict(color=color, dash=dash, width=2.5)
        ))

    # 4. Layout configuration
    fig.update_layout(
        title="<b>Task 1: Structural Probability Density Functions of the Beta Distribution</b>",
        xaxis_title="<b>CTR / Click Probability (θ)</b>",
        yaxis_title="<b>Probability Density</b>",
        template="plotly_white",
        legend=dict(x=0.02, y=0.98),
        hovermode="x unified",
        xaxis=dict(range=[0, 1])
    )

    fig.show()

# Execute plot
plot_beta_ctr_priors()

---
## 2: Sequential Likelihood and Joint History

### 1. Likelihood Contribution of a Single Isolated Response $y_k$
Let $y_k \in \{0, 1\}$ denote the observed user interaction with the advertisement at time step $k$, where $y_k = 1$ indicates a click and $y_k = 0$ indicates a non-click[cite: 1].

Conditional on the true, unknown click-through rate $\Theta = \theta$ (where $\theta \in [0, 1]$), each user interaction is modeled as an independent Bernoulli trial[cite: 1]:

$$P(Y_k = 1 \mid \Theta = \theta) = \theta$$
$$P(Y_k = 0 \mid \Theta = \theta) = 1 - \theta$$

The mathematical likelihood contribution $L(y_k \mid \theta)$ of a single isolated response at step $k$ is expressed as the Bernoulli probability mass function:

$$L(y_k \mid \theta) = P(Y_k = y_k \mid \Theta = \theta) = \theta^{y_k} (1 - \theta)^{1 - y_k}$$


### 2. Joint Likelihood Function for the Running History Vector $y^{(k)}$
Let $y^{(k)} = (y_1, y_2, \dots, y_k)$ denote the running vector of observed user interactions accumulated up to impression step $k$ (where $1 \le k \le n$)[cite: 1].

Assuming each individual user interaction is conditionally independent given the true conversion rate $\theta$, the joint likelihood function $L\left(y^{(k)} \mid \theta\right)$ is the product of the individual Bernoulli likelihood contributions from step $1$ to step $k$[cite: 1]:

$$L\left(y^{(k)} \mid \theta\right) = P\left(Y^{(k)} = y^{(k)} \mid \Theta = \theta\right) = \prod_{i=1}^k L(y_i \mid \theta) = \prod_{i=1}^k \theta^{y_i} (1 - \theta)^{1 - y_i}$$

By combining terms with identical bases using standard exponent rules, the joint likelihood simplifies to:

$$L\left(y^{(k)} \mid \theta\right) = \theta^{\sum_{i=1}^k y_i} (1 - \theta)^{\sum_{i=1}^k (1 - y_i)}$$

Let $S_k = \sum_{i=1}^k y_i$ represent the total number of observed clicks up to step $k$, and let $F_k = k - S_k = \sum_{i=1}^k (1 - y_i)$ represent the total number of observed non-clicks (failures) up to step $k$. The joint likelihood function can be compactly written as:

$$L\left(y^{(k)} \mid \theta\right) = \theta^{S_k} (1 - \theta)^{k - S_k}$$

---
## 3: Closed-Form Analytical Updates (Beta-Binomial Conjugacy)

### 1. Derivation of the Recursive Posterior Density
By Bayes' Theorem, the posterior density of the true click-through rate parameter $\Theta = \theta$ at step $k$, given the running interaction vector $y^{(k)} = (y_1, y_2, \dots, y_k)$, is proportional to the product of the likelihood contribution $L(y_k \mid \theta)$ and the posterior density from step $k-1$ (which acts as the prior state for step $k$)[cite: 1]:

$$f_{\Theta \mid Y^{(k)}}\left(\theta \mid y^{(k)}\right) \propto L(y_k \mid \theta) \cdot f_{\Theta \mid Y^{(k-1)}}\left(\theta \mid y^{(k-1)}\right)$$

Assume at step $k-1$, the posterior distribution follows a Beta density with shape parameters $\alpha_{k-1}$ and $\beta_{k-1}$[cite: 1]:

$$f_{\Theta \mid Y^{(k-1)}}\left(\theta \mid y^{(k-1)}\right) = \frac{1}{B(\alpha_{k-1}, \beta_{k-1})} \theta^{\alpha_{k-1} - 1} (1 - \theta)^{\beta_{k-1} - 1}$$

From Task 2, the single-step likelihood contribution of observation $y_k \in \{0, 1\}$ is[cite: 1]:

$$L(y_k \mid \theta) = \theta^{y_k} (1 - \theta)^{1 - y_k}$$

Multiplying the likelihood and prior density together:

$$f_{\Theta \mid Y^{(k)}}\left(\theta \mid y^{(k)}\right) \propto \left[ \theta^{y_k} (1 - \theta)^{1 - y_k} \right] \cdot \left[ \theta^{\alpha_{k-1} - 1} (1 - \theta)^{\beta_{k-1} - 1} \right]$$

Combining powers of $\theta$ and $(1 - \theta)$:

$$f_{\Theta \mid Y^{(k)}}\left(\theta \mid y^{(k)}\right) \propto \theta^{(\alpha_{k-1} + y_k) - 1} (1 - \theta)^{(\beta_{k-1} + 1 - y_k) - 1}$$


### 2. Proof of Beta-Binomial Conjugacy
Notice that the unnormalized posterior density takes the exact functional form of a Beta distribution kernel, $\theta^{\alpha_k - 1} (1 - \theta)^{\beta_k - 1}$[cite: 1].

Because the posterior density remains in the same parametric probability family as the prior, the Beta distribution is a **conjugate prior** for the Bernoulli/Binomial likelihood[cite: 1].

The closed-form arithmetic update equations for the parameters at step $k$ are:

$$\alpha_k = \alpha_{k-1} + y_k$$

$$\beta_k = \beta_{k-1} + (1 - y_k)$$

* If a user **clicks** ($y_k = 1$), the success parameter increments by 1 ($\alpha_k = \alpha_{k-1} + 1$, $\beta_k = \beta_{k-1}$).
* If a user **does not click** ($y_k = 0$), the failure parameter increments by 1 ($\alpha_k = \alpha_{k-1}$, $\beta_k = \beta_{k-1} + 1$).


### 3. Posterior Mean at Time Step $k$
Using the properties of the updated $\text{Beta}(\alpha_k, \beta_k)$ distribution, the Expected A Posteriori (EAP) estimate for the latent conversion rate $\Theta$ is[cite: 1]:

$$\mathbb{E}\left[\Theta \mid Y^{(k)} = y^{(k)}\right] = \frac{\alpha_k}{\alpha_k + \beta_k} = \frac{\alpha_0 + \sum_{i=1}^k y_i}{\alpha_0 + \beta_0 + k}$$

---
## 4: Dynamic Shifting Mechanics (Conjugate vs. Non-Conjugate Updates)

### 1. Mathematical Shifting Dynamics of Clicks vs. Non-Clicks
Under the Beta-Binomial conjugate model, observing a user interaction $y_k \in \{0, 1\}$ updates the shape of the density function instantly through simple scalar additions to the shape parameters $\alpha$ and $\beta$[cite: 1]:

* **Observed Click ($y_k = 1$):**
  * Parameter Update: $\alpha_k = \alpha_{k-1} + 1$, $\beta_k = \beta_{k-1}$[cite: 1].
  * Density Effect: The numerator of the posterior mode $\frac{\alpha_k - 1}{\alpha_k + \beta_k - 2}$ increases while $\beta$ remains unchanged[cite: 1]. This shifts the entire center of mass and the peak of the probability density **rightward** toward $\theta = 1.0$[cite: 1], increasing the platform's estimate of the advertisement's CTR.

* **Observed Non-Click ($y_k = 0$):**
  * Parameter Update: $\alpha_k = \alpha_{k-1}$, $\beta_k = \beta_{k-1} + 1$[cite: 1].
  * Density Effect: The denominator increases while the success count $\alpha$ stays constant[cite: 1]. This pulls the ratio down and shifts the peak of the density **leftward** toward $\theta = 0.0$[cite: 1], penalizing the estimated conversion rate.

In both cases, total pseudo-counts increase ($\alpha_k + \beta_k = \alpha_0 + \beta_0 + k$)[cite: 1]. This causes the variance $\text{Var}(\Theta) = \frac{\alpha_k \beta_k}{(\alpha_k + \beta_k)^2(\alpha_k + \beta_k + 1)}$ to shrink continuously, making the distribution **narrower and sharper** around its peak as empirical evidence accumulates[cite: 1].

### 2. Analytical Conjugacy vs. Non-Conjugate Numerical Grid Frameworks

| Aspect | Beta-Binomial Conjugate Model (CTR) | 2PL IRT Non-Conjugate Model (Ability) |
| :--- | :--- | :--- |
| **Prior-Likelihood Relationship** | **Conjugate:** Beta prior paired with Bernoulli/Binomial likelihood yields a closed-form Beta posterior[cite: 1]. | **Non-Conjugate:** Gaussian prior paired with non-linear logistic IRF ($1 / (1 + e^{-a(\theta-b)})$\)) has no closed-form solution[cite: 1]. |
| **Computational Mechanism** | **Exact Arithmetic:** Updated instantly via simple additions ($\alpha_k = \alpha_{k-1} + y_k$)[cite: 1]. | **Numerical Quadrature/Grid:** Requires point-wise array multiplication across hundreds/thousands of discrete grid points[cite: 1]. |
| **Normalization Method** | **Analytical Constant:** Normalized automatically using the Beta function $B(\alpha_k, \beta_k)$[cite: 1]. | **Numerical Integration:** Requires explicit step-by-step trapezoidal integration (`np.trapz`)[cite: 1]. |
| **Resource Efficiency** | $O(1)$ memory and execution time per step. Ideal for real-time high-throughput streaming systems[cite: 1]. | $O(M)$ operations per step, where $M$ is the number of grid points[cite: 1]. |

---
## 5: Running Point Estimators (Closed-Form Equations)

Under the conjugate Beta-Binomial framework, the posterior distribution at time step $k$ is known analytically to be $\text{Beta}(\alpha_k, \beta_k)$, where $\alpha_k = \alpha_0 + \sum_{i=1}^k y_i$ and $\beta_k = \beta_0 + k - \sum_{i=1}^k y_i$.

We evaluate point estimates directly from these updated shape parameters using simple, exact arithmetic expressions without numerical integration.


### 1. Running Posterior Mean ($\hat{\theta}_{\text{Bayes}}^{(k)}$)
The Expected A Posteriori (EAP) estimate represents the expected value (first moment) of the updated $\text{Beta}(\alpha_k, \beta_k)$ density:

$$\hat{\theta}_{\text{Bayes}}^{(k)} = \mathbb{E}\left[\Theta \mid Y^{(k)} = y^{(k)}\right] = \frac{\alpha_k}{\alpha_k + \beta_k}$$

In terms of initial prior parameters $(\alpha_0, \beta_0)$ and cumulative observed counts:

$$\hat{\theta}_{\text{Bayes}}^{(k)} = \frac{\alpha_0 + S_k}{\alpha_0 + \beta_0 + k}$$

where $S_k = \sum_{i=1}^k y_i$ is the total number of clicks observed up to step $k$.


### 2. Running Maximum A Posteriori ($\hat{\theta}_{\text{MAP}}^{(k)}$)
The Maximum A Posteriori (MAP) estimate represents the mode (highest peak) of the updated $\text{Beta}(\alpha_k, \beta_k)$ posterior density.

For $\alpha_k > 1$ and $\beta_k > 1$, the mode is given by:

$$\hat{\theta}_{\text{MAP}}^{(k)} = \arg\max_{\theta \in [0, 1]} f_{\Theta \mid Y^{(k)}}\left(\theta \mid y^{(k)}\right) = \frac{\alpha_k - 1}{\alpha_k + \beta_k - 2}$$

In terms of initial prior parameters $(\alpha_0, \beta_0)$ and cumulative observed counts:

$$\hat{\theta}_{\text{MAP}}^{(k)} = \frac{\alpha_0 + S_k - 1}{\alpha_0 + \beta_0 + k - 2}$$

*(Note: If $\alpha_k = 1$ and $\beta_k = 1$, the distribution is flat/uniform, and any $\theta \in [0, 1]$ is a mode).*

In [ ]:
import numpy as np
import plotly.graph_objects as go
from scipy.stats import beta

def plot_beta_density_progression(
    theta_true=0.35,
    n_impressions=100,
    alpha_0=1,
    beta_0=1,
    seed=42
):
    """
    Simulates $n=100$ Bernoulli impressions and plots the progressive
    evolution of the Beta density curves at key milestone steps.
    """
    np.random.seed(seed)

    # 1. Define continuous domain theta in [0, 1]
    theta = np.linspace(0.0, 1.0, 1000)

    # Milestones to visualize
    milestones = [0, 1, 2, 5, 10, 30, 50, 100]

    # Store parameter history and event labels
    history = {}

    alpha_k = alpha_0
    beta_k = beta_0

    # Step 0 initial prior
    history[0] = {
        'alpha': alpha_k,
        'beta': beta_k,
        'label': f'Initial Prior: Beta({alpha_k},{beta_k})',
        'color': 'gray',
        'dash': 'dash'
    }

    # Colors for subsequent milestones
    colors = {
        1: '#E45756',   # Red-orange
        2: '#2CA02C',   # Green
        5: '#9467BD',   # Purple
        10: '#FF7F0E',  # Orange
        30: '#17BECF',  # Cyan
        50: '#E377C2',  # Pink
        100: '#BCBD22'  # Yellow-green
    }

    # 2. Run simulation and record milestones
    for k in range(1, n_impressions + 1):
        u = np.random.uniform(0.0, 1.0)
        y_k = 1 if u < theta_true else 0

        event_str = "Click" if y_k == 1 else "No Click"

        alpha_k += y_k
        beta_k += (1 - y_k)

        if k in milestones:
            history[k] = {
                'alpha': alpha_k,
                'beta': beta_k,
                'label': f'Step {k}: After Event ({event_str}, a={alpha_k}, β={beta_k})',
                'color': colors[k],
                'dash': 'solid'
            }

    # 3. Construct Plotly Figure
    fig = go.Figure()

    for k in milestones:
        item = history[k]
        pdf_vals = beta.pdf(theta, item['alpha'], item['beta'])

        fig.add_trace(go.Scatter(
            x=theta,
            y=pdf_vals,
            mode='lines',
            name=item['label'],
            line=dict(color=item['color'], dash=item['dash'], width=2)
        ))

    # Vertical Reference Line at theta_true
    fig.add_vline(
        x=theta_true,
        line_width=2,
        line_dash="dot",
        line_color="red",
        annotation_text=f"True CTR ({theta_true})",
        annotation_position="top right"
    )

    # 4. Styling & Layout Configuration
    fig.update_layout(
        title={
            'text': "<b>Analytical Beta-Binomial Conjugate Update Timeline</b>",
            'y': 0.05,
            'x': 0.5,
            'xanchor': 'center',
            'yanchor': 'bottom',
            'font': dict(size=16)
        },
        xaxis_title="<b>Conversion Rate Parameter (θ)</b>",
        yaxis_title="<b>Probability Density f(θ | y)</b>",
        template="plotly_white",
        legend=dict(x=0.65, y=0.98),
        hovermode="x unified",
        margin=dict(b=80, t=40)
    )

    fig.show()

# Run and show plot
plot_beta_density_progression()

In [ ]:
import numpy as np
import plotly.graph_objects as go

def track_beta_binomial_ctr_convergence(
    theta_true=0.35,
    n_impressions=100,
    alpha_0=1,
    beta_0=1,
    seed=42
):
    """
    Task 6: Tracks closed-form Beta-Binomial sequential estimators (Posterior Mean
    and MAP) over n=100 impression steps and visualizes convergence against theta_true.
    """
    np.random.seed(seed)

    # Initialize tracking lists
    # Step 0 prior states
    alpha_k = alpha_0
    beta_k = beta_0

    bayes_history = [alpha_k / (alpha_k + beta_k)]
    # MAP for uniform Beta(1,1) is undefined/flat, initialized at mean 0.5
    map_history = [0.5]

    # Simulate sequential updates across n_impressions
    for k in range(1, n_impressions + 1):
        # Dynamically generate response y_k in {0, 1}
        u = np.random.uniform(0.0, 1.0)
        y_k = 1 if u < theta_true else 0

        # Closed-form parameter updates
        alpha_k += y_k
        beta_k += (1 - y_k)

        # Calculate running point estimates
        # Running Posterior Mean (EAP / Bayes)
        theta_bayes = alpha_k / (alpha_k + beta_k)

        # Running Maximum A Posteriori (MAP)
        if alpha_k > 1 and beta_k > 1:
            theta_map = (alpha_k - 1) / (alpha_k + beta_k - 2)
        else:
            theta_map = theta_bayes

        bayes_history.append(theta_bayes)
        map_history.append(theta_map)

    # Generate Interactive Plotly Chart
    steps = list(range(n_impressions + 1))
    fig = go.Figure()

    # Posterior Mean Curve
    fig.add_trace(go.Scatter(
        x=steps, y=bayes_history,
        mode='lines',
        name='Posterior Mean (θ_Bayes)',
        line=dict(color='#1f77b4', width=2.5)
    ))

    # MAP Estimate Curve
    fig.add_trace(go.Scatter(
        x=steps, y=map_history,
        mode='lines',
        name='MAP Estimate (θ_MAP)',
        line=dict(color='#d62728', width=2, dash='dot')
    ))

    # Static Horizontal Reference Line at theta_true
    fig.add_hline(
        y=theta_true, line_width=2, line_dash="dash", line_color="black",
        annotation_text=f"True CTR (θ_true = {theta_true})",
        annotation_position="bottom right"
    )

    fig.update_layout(
        title="<b>Task 6: Sequential Convergence of CTR Estimators (n = 100 Impressions)</b>",
        xaxis_title="<b>Impression Step (k)</b>",
        yaxis_title="<b>Estimated Click-Through Rate (θ)</b>",
        template="plotly_white",
        legend=dict(x=0.65, y=0.98),
        hovermode="x unified",
        yaxis=dict(range=[0, 1])
    )

    fig.show()

# Run tracking simulation
track_beta_binomial_ctr_convergence()